# 05 分布式通信原语

## 简介

本笔记本介绍分布式训练中最基础的通信原语：broadcast、all-reduce、all-gather、reduce-scatter。
这些原语是数据并行、张量并行、流水线并行等策略的通信基石。

我们将通过我们的手写实现（基于 `isend/irecv`）来理解每种原语的通信模式和复杂度。

注意：多进程通信需要 `torchrun` 启动，本笔记本以概念演示和公式分析为主。

## 1. 导入

In [ ]:
import torch
import torch.distributed as dist
import os

from parallel.communication.setup import get_rank, get_world_size, init_process_group, cleanup
from parallel.communication.topologies import (
    analyze_ring_topology,
    analyze_tree_topology,
    analyze_mesh_topology,
    visualize_topology,
)

print(f"当前 rank (单进程模式): {get_rank()}")
print(f"world_size (单进程模式): {get_world_size()}")
print(f"dist 已初始化: {dist.is_initialized()}")

## 2. 四种基本集合通信操作

| 操作 | 方向 | 效果 | 复杂度 (Ring) |
|---|---|---|---|
| **Broadcast** | 1 → N | 源 rank 数据发送到所有 rank | 2N, O(log P) (树) |
| **All-Reduce** | N → 1 → N | 所有 rank 数据求和/平均后广播 | 2N, O(2N/P) per step |
| **All-Gather** | N → N | 拼接所有 rank 的数据到每个 rank | (P-1)N/P per step |
| **Reduce-Scatter** | N → N/P | 规约后每个 rank 保留 1/P | (P-1)N/P per step |

## 3. Broadcast 详解

Broadcast 将源 rank 的数据复制到所有其他 rank。
朴素实现: rank 0 依次发送给 rank 1, 2, 3... (O(P))
树形拓扑: rank 0 发给子节点，子节点再转发 (O(log P))

In [ ]:
print("=" * 60)
print("Broadcast 通信模式")
print("=" * 60)
print()
print("朴素 (Linear):  O(P) 步")
print("  Rank 0 -> Rank 1 -> Rank 2 -> Rank 3")
print()
print("树形 (Tree):    O(log P) 步")
print("  Rank 0 -> {Rank 1, Rank 2}")
print("  Rank 1 -> {Rank 3, Rank 4}")
print("  Rank 2 -> {Rank 5, Rank 6}")

print(f"\nP=8 时:")
tree = analyze_tree_topology(world_size=8)
print(f"  树高: {tree['tree_height']}")
print(f"  Broadcast 步数: {tree['total_steps_for_broadcast']}")
print(f"  {tree['description']}")

### 🧠 直觉理解：集合通信

集合通信就像团队协作中的不同沟通模式：
- **Broadcast** = 老板向全员发通知（1→N）
- **All-Reduce** = 全员汇总意见后共享结果（N→N，先汇总再广播）
- **All-Gather** = 每人分享自己的部分，全员获得完整信息（N→N，拼接）
- **Reduce-Scatter** = 汇总后每人只拿自己负责的部分（N→N/P）

**Ring All-Reduce 的直觉**：想象 4 个人围坐一圈，每人手里有 4 份报告的不同部分。第一轮每人把自己的一份传给右边的人并累加收到的一份，4 轮后每人手里有一份完整的汇总；第二轮再把汇总结果传一圈，最终每人都有完整的 4 份汇总。

### 📐 数学原理：Ring All-Reduce 通信复杂度

设数据总量为 $N$，进程数为 $P$，带宽为 $B$，延迟为 $L$。

**Ring All-Reduce 分两阶段**：

1. **Reduce-Scatter** ($P-1$ 步)：
$$T_{rs} = (P-1) \times \left(L + \frac{N/P}{B}\right)$$

2. **All-Gather** ($P-1$ 步)：
$$T_{ag} = (P-1) \times \left(L + \frac{N/P}{B}\right)$$

**总时间**：
$$T_{total} = 2(P-1) \times \left(L + \frac{N}{PB}\right)$$

**关键性质**：总通信量恒为 $2N$，与 $P$ 无关（带宽最优）。增加 GPU 只影响延迟项。

## 4. All-Reduce 详解 (Ring 算法)

Ring All-Reduce 是分布式训练中最核心的通信原语。
它通过环形拓扑在 2(P-1) 步内完成规约，总通信量恒为 2N，与进程数无关。

**阶段一 (Reduce-Scatter)**: P-1 步，每步向右邻居发送一个分片并累加左邻居发来的分片
**阶段二 (All-Gather)**: P-1 步，每步向右邻居发送一个已规约分片并接收左邻居的新分片

In [ ]:
print("=" * 60)
print("Ring All-Reduce 逐步演示 (P=4)")
print("=" * 60)

# 模拟 4 个 rank, 每个持有 4 个分片
P = 4
chunks_per_rank = 4

# 初始状态: 各 rank 拥有总数据 1/P
print("\n初始状态 (数据分片):")
for rank in range(P):
    chunks_str = " | ".join(f"D{r}{c}" for c in range(chunks_per_rank))
    print(f"  Rank {rank}: [{chunks_str}]")

print(f"\n阶段一: Reduce-Scatter ({P}-1 = {P-1} 步)")
print("  每步: 发送一个分片给右邻居，接收左邻居的分片并累加")
print(f"  Step 1: Rank{r} 发送 chunk_{r} -> 右邻居")
print(f"  ... 经过 {P-1} 步后，每个 rank 持有一个完整的规约分片")

print(f"\n阶段二: All-Gather ({P}-1 = {P-1} 步)")
print("  每步: 发送已规约分片给右邻居，接收左邻居的新分片并覆盖")
print(f"  ... 经过 {P-1} 步后，所有 rank 拥有全部规约结果")

# 分析
ring = analyze_ring_topology(world_size=4)
print(f"\nRing All-Reduce (P=4) 分析:")
print(f"  Reduce-Scatter: {ring['steps_reduce_scatter']} 步")
print(f"  All-Gather:     {ring['steps_all_gather']} 步")
print(f"  总步数:         {ring['total_steps']} 步")
print(f"  每步数据量:     {ring['per_step_data']}")

In [ ]:
# Ring All-Reduce ASCII 图
print("Ring All-Reduce 通信拓扑:")
print()
for P in [4, 8]:
    visualize_topology(P, "ring")
    print()

### 📊 Ring All-Reduce 两阶段步骤可视化

用 matplotlib 可视化 Ring All-Reduce 的两个阶段中数据如何在各 rank 之间传递。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

P = 4
ranks = list(range(P))

# 左图: Reduce-Scatter 阶段
ax = axes[0]
for step in range(P - 1):
    for rank in range(P):
        send_chunk = (rank - step) % P
        recv_chunk = (rank - step - 1) % P
        target = (rank + 1) % P
        ax.annotate('', xy=(target, step + 0.3), xytext=(rank, step + 0.3),
                    arrowprops=dict(arrowstyle='->', color=f'C{send_chunk}', lw=2))
        ax.text(rank + 0.5, step + 0.3, f'C{send_chunk}', fontsize=8, ha='center', va='center')

for rank in range(P):
    ax.plot(rank, -0.3, 'o', markersize=15, color='steelblue')
    ax.text(rank, -0.3, f'R{rank}', ha='center', va='center', fontsize=10, fontweight='bold', color='white')

ax.set_xlim(-0.5, P - 0.5)
ax.set_ylim(-0.8, P - 1.5)
ax.set_xlabel('Rank', fontsize=12)
ax.set_ylabel('步骤', fontsize=12)
ax.set_title('Reduce-Scatter 阶段\n(每步发送一个分片给右邻居)', fontsize=13)
ax.set_yticks(range(P - 1))

# 右图: All-Gather 阶段
ax2 = axes[1]
for step in range(P - 1):
    for rank in range(P):
        send_chunk = (rank - step) % P
        target = (rank + 1) % P
        ax2.annotate('', xy=(target, step + 0.3), xytext=(rank, step + 0.3),
                     arrowprops=dict(arrowstyle='->', color=f'C{send_chunk}', lw=2))
        ax2.text(rank + 0.5, step + 0.3, f'C{send_chunk}', fontsize=8, ha='center', va='center')

for rank in range(P):
    ax2.plot(rank, -0.3, 'o', markersize=15, color='steelblue')
    ax2.text(rank, -0.3, f'R{rank}', ha='center', va='center', fontsize=10, fontweight='bold', color='white')

ax2.set_xlim(-0.5, P - 0.5)
ax2.set_ylim(-0.8, P - 1.5)
ax2.set_xlabel('Rank', fontsize=12)
ax2.set_ylabel('步骤', fontsize=12)
ax2.set_title('All-Gather 阶段\n(每步发送已规约分片给右邻居)', fontsize=13)
ax2.set_yticks(range(P - 1))

plt.tight_layout()
plt.show()

## 5. All-Gather 和 Reduce-Scatter

- **All-Gather**: 拼接所有 rank 的数据，每个 rank 得到完整的 N×P 数据（即 Ring All-Reduce 的阶段二）
- **Reduce-Scatter**: 规约后切成 P 份，每个 rank 保留 1/P 份（即 Ring All-Reduce 的阶段一）

In [ ]:
print("All-Gather 示意:")
print("  输入: 每个 rank 持有 N/P 的数据")
print("  输出: 每个 rank 持有全部 N 的数据 (拼接在一起)")
print()
for rank in range(4):
    chunks_str = " | ".join(f"D{r}{c}" for c in range(2))
    gathered_str = " | ".join(f"D{rr}{c}" for rr in range(4) for c in range(2))
    print(f"  Rank {rank}: 本地[{chunks_str}] -> AllGather -> [{gathered_str}]")

print("\nReduce-Scatter 示意:")
print("  输入: 每个 rank 持有全部 N 的数据")
print("  输出: N 被分为 P 份，每个 rank 持有一份规约后的结果")

print("\n★ All-Reduce = Reduce-Scatter + All-Gather")
print("★ 这正是 Ring All-Reduce 算法将全规约分解为两个阶段的原因")

## 6. 通信代价公式

假设数据量为 N，带宽为 B，延迟为 L，进程数为 P：

| 操作 | 朴素实现 | Ring 实现 |
|---|---|---|
| **Broadcast** | L × P + N/B × P | L × log₂P + N/B × log₂P (树) |
| **All-Reduce** | L × P + N/B × P | 2 × (L × (P-1) + N/B × (P-1)/P) |
| **All-Gather** | L × P + N/B × P | L × (P-1) + N/B × (P-1)/P |
| **Reduce-Scatter** | L × P + N/B × P | L × (P-1) + N/B × (P-1)/P |

In [ ]:
def comm_cost_ring(operation, N_gb, P, bandwidth_gbps=100, latency_ms=0.01):
    """计算 Ring 算法的通信时间 (ms)"""
    # N 转换为 bits
    N_bits = N_gb * 8

    if operation == "allreduce":
        steps = 2 * (P - 1)
        data_per_step = N_gb / P  # GB per step
    elif operation == "broadcast":
        steps = 1
        data_per_step = N_gb
    else:
        steps = P - 1
        data_per_step = N_gb / P

    transfer_time = data_per_step / (bandwidth_gbps / 8) * 1000  # ms
    latency = latency_ms * steps
    total = latency + transfer_time * steps
    return total

# 场景: 1GB 梯度，100Gbps 带宽，不同 GPU 数
print("Ring All-Reduce 通信时间 (N=1GB, B=100Gbps):")
for P in [4, 8, 16, 32, 64]:
    t = comm_cost_ring("allreduce", 1.0, P)
    print(f"  P={P:3d}: {t:.2f} ms")

print("\n★ 注意: Ring 算法总通信量恒为 2N，与 P 无关!")
print("  但每步传输量随 P 增加而减小 (N/P)，延迟项随 P 线性增加")

### 📊 通信量对比柱状图

对比 AllReduce、AllGather、ReduceScatter 在不同 GPU 数量下的通信时间。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

def comm_time(operation, N_gb, P, B_gbps=100, L_ms=0.01):
    if operation == 'allreduce':
        steps = 2 * (P - 1)
        data_per_step = N_gb / P
    elif operation == 'allgather':
        steps = P - 1
        data_per_step = N_gb / P
    else:  # reduce_scatter
        steps = P - 1
        data_per_step = N_gb / P
    transfer = data_per_step / (B_gbps / 8) * 1000
    return L_ms * steps + transfer * steps

P_values = [4, 8, 16, 32, 64]
N_gb = 1.0

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(P_values))
width = 0.25

ar_times = [comm_time('allreduce', N_gb, P) for P in P_values]
ag_times = [comm_time('allgather', N_gb, P) for P in P_values]
rs_times = [comm_time('reduce_scatter', N_gb, P) for P in P_values]

ax.bar(x - width, ar_times, width, label='All-Reduce', color='#e74c3c', edgecolor='black', linewidth=0.5)
ax.bar(x, ag_times, width, label='All-Gather', color='#3498db', edgecolor='black', linewidth=0.5)
ax.bar(x + width, rs_times, width, label='Reduce-Scatter', color='#2ecc71', edgecolor='black', linewidth=0.5)

ax.set_xlabel('GPU 数量', fontsize=12)
ax.set_ylabel('通信时间 (ms)', fontsize=12)
ax.set_title(f'通信量对比 (N={N_gb}GB, B=100Gbps)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([f'P={P}' for P in P_values])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Megatron 风格 2D 网格拓扑

实际大模型训练常组合张量并行 (TP) 和数据并行 (DP)，形成二维通信网格。

In [ ]:
# 二维网格分析
for tp, dp in [(4, 2), (8, 2), (8, 4)]:
    mesh = analyze_mesh_topology(tp_size=tp, dp_size=dp)
    print(f"TP={tp}, DP={dp} ({mesh['total_devices']} GPUs):")
    print(f"  TP 通信: {mesh['tp_communication_steps']} 步")
    print(f"  DP 通信: {mesh['dp_communication_steps']} 步")
    print(f"  每层总通信: {mesh['communication_cost_per_layer']} 步")
    print()

print("Mesh 拓扑图 (TP=2, DP=4, 8 GPUs):")
visualize_topology(8, "mesh_2x4")
print("\n行方向 = TP 组 (同一样本的不同分片)")
print("列方向 = DP 组 (不同样本, 同步梯度)")

## 8. 关键要点总结

1. **Broadcast**: 1 到 N 的数据分发，树形拓扑可实现 O(log P) 复杂度
2. **All-Reduce**: 所有 rank 求和/平均，Ring 算法总通信量 2N，与 P 无关
3. **All-Reduce = Reduce-Scatter + All-Gather**: 两个阶段各 P-1 步
4. **Ring All-Reduce** 是 NCCL/Horovod 的核心算法，带宽最优
5. **通信开销 = 延迟 × 步数 + 传输时间**: 小消息受延迟主导，大消息受带宽主导
6. **现实世界中**更多 GPU 并不会线性增加 All-Reduce 时间 (得益于 N/P 分片)

## 📝 练习题

### 思考题
1. 为什么 Ring All-Reduce 的总通信量恒为 2N，与 GPU 数量 P 无关？这个性质在什么条件下成立？

### 编程题
2. 编写一个函数，模拟 Ring All-Reduce 的 Reduce-Scatter 阶段：给定 4 个 rank 各持有 [1,2,3,4]，经过 3 步后每个 rank 应持有一个完整的规约分片。打印每步后各 rank 的数据状态。

### 分析题
3. 在 100Gbps 带宽、0.01ms 延迟的网络下，对于 7B 参数模型（fp16 约 14GB 梯度），计算 P=8 和 P=64 时的 All-Reduce 通信时间。分析更多 GPU 是否总是更快？